# DiT Model Testing Suite

Comprehensive testing notebook for the Diffusion Transformer (DiT) model, covering forward passes, gradients, conditioning effects, and overfitting validation.

In [1]:
import torch
from models.dit import DiT

film_conditioner.py STARTED
ABOUT TO DEFINE FiLMConditioner


In [9]:


# Initialize model
model = DiT(
    patch_dim=256,
    embed_dim=512,
    num_blocks=2,  # keep small for tests
    num_heads=8,
    num_genres=10,
)

# Configuration
PATCH_DIM = 256
SEQ_LEN = 16
NUM_GENRES = 10
DEVICE = "cpu"
BATCH_SIZE = 4

model = model.to(DEVICE)
print("✓ Model initialized and moved to device")

✓ Model initialized and moved to device


## Helper Functions

Create dummy inputs for testing with configurable batch sizes and gradient requirements.

In [10]:
def create_dummy_inputs(batch_size=4, requires_grad=False):
    """Generate random inputs matching the model's expected format."""
    x = torch.randn(
        batch_size,
        SEQ_LEN,
        PATCH_DIM,
        device=DEVICE,
        requires_grad=requires_grad,
    )
    t = torch.rand(batch_size, device=DEVICE)
    genre_ids = torch.randint(
        0, NUM_GENRES, (batch_size,), device=DEVICE
    )
    return x, t, genre_ids

## Test 1: Forward Pass

Verify that the model produces outputs with the correct shape.

In [11]:
print("▶ Forward pass test")
x, t, genre_ids = create_dummy_inputs()

with torch.no_grad():
    y = model(x, t, genre_ids)

assert y.shape == x.shape, f"Output shape {y.shape} != input {x.shape}"
print(f"  ✓ Forward pass OK")
print(f"    Input shape:  {x.shape}")
print(f"    Output shape: {y.shape}")

▶ Forward pass test
  ✓ Forward pass OK
    Input shape:  torch.Size([4, 16, 256])
    Output shape: torch.Size([4, 16, 256])


## Test 2: Gradient Flow

Ensure gradients are properly computed for all trainable parameters.

In [5]:
print("▶ Gradient flow test")
x, t, genre_ids = create_dummy_inputs(requires_grad=True)

y = model(x, t, genre_ids)
loss = y.mean()
loss.backward()

no_grad_params = []
for name, param in model.named_parameters():
    if param.requires_grad and param.grad is None:
        no_grad_params.append(name)

if no_grad_params:
    print(f"  ✗ No gradients for: {no_grad_params}")
else:
    print("  ✓ Gradients OK - all parameters have gradients")

▶ Gradient flow test
  ✓ Gradients OK - all parameters have gradients


## Test 3: Conditioning Effects

Verify that both timestep and genre conditioning influence the model outputs.

In [6]:
print("▶ Conditioning influence test")
x, _, _ = create_dummy_inputs()

t0 = torch.zeros(x.size(0), device=DEVICE)
t1 = torch.ones(x.size(0), device=DEVICE)

g0 = torch.zeros(x.size(0), dtype=torch.long, device=DEVICE)
g1 = torch.ones(x.size(0), dtype=torch.long, device=DEVICE)

with torch.no_grad():
    y_t0 = model(x, t0, g0)
    y_t1 = model(x, t1, g0)
    y_g1 = model(x, t0, g1)

t_diff = (y_t0 - y_t1).abs().mean().item()
g_diff = (y_t0 - y_g1).abs().mean().item()

assert t_diff > 1e-5, "Timestep conditioning has no effect"
assert g_diff > 1e-5, "Genre conditioning has no effect"

print(f"  ✓ Timestep conditioning difference: {t_diff:.6f}")
print(f"  ✓ Genre conditioning difference:    {g_diff:.6f}")

▶ Conditioning influence test
  ✓ Timestep conditioning difference: 0.060575
  ✓ Genre conditioning difference:    0.057408


## Test 4: Output Statistics

Check for NaN values and ensure the output has non-zero variance.

In [7]:
print("▶ Output statistics test")
x, t, genre_ids = create_dummy_inputs()

with torch.no_grad():
    y = model(x, t, genre_ids)

mean = y.mean().item()
std = y.std().item()

assert not torch.isnan(y).any(), "NaNs detected in output"
assert std > 0, "Output variance collapsed"

print(f"  ✓ Mean: {mean:.6f}")
print(f"  ✓ Std:  {std:.6f}")
print(f"  ✓ No NaN values detected")

▶ Output statistics test
  ✓ Mean: -0.001487
  ✓ Std:  0.578200
  ✓ No NaN values detected


## Test 5: Tiny Batch Overfitting

Verify that the model can overfit to a tiny batch, demonstrating learning capacity.

In [8]:
print("▶ Tiny batch overfitting test")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

x, t, genre_ids = create_dummy_inputs(batch_size=2)
target = torch.randn_like(x)

initial_loss = None
losses = []

steps = 200
for step in range(steps):
    optimizer.zero_grad()
    y = model(x, t, genre_ids)
    loss = ((y - target) ** 2).mean()

    if step == 0:
        initial_loss = loss.item()

    loss.backward()
    optimizer.step()
    
    if step % 50 == 0 or step == steps - 1:
        losses.append((step, loss.item()))

final_loss = loss.item()
assert final_loss < initial_loss * 0.5, "Model failed to overfit tiny batch"

print(f"  ✓ Loss decreased from {initial_loss:.6f} to {final_loss:.6f}")
print(f"  ✓ Reduction: {(1 - final_loss/initial_loss)*100:.1f}%")
print("\n  Loss progression:")
for step, l in losses:
    print(f"    Step {step:3d}: {l:.6f}")

▶ Tiny batch overfitting test
  ✓ Loss decreased from 1.328280 to 0.000607
  ✓ Reduction: 100.0%

  Loss progression:
    Step   0: 1.328280
    Step  50: 0.007271
    Step 100: 0.001932
    Step 150: 0.000962
    Step 199: 0.000607


In [10]:
x = torch.randn(BATCH_SIZE, SEQ_LEN, PATCH_DIM)
t = torch.linspace(0, 1, BATCH_SIZE)
genre_ids = torch.randint(0, NUM_GENRES, (BATCH_SIZE,))

y = model(x, t, genre_ids)
print(y.std(), y.mean())

tensor(0.6919, grad_fn=<StdBackward0>) tensor(-0.0072, grad_fn=<MeanBackward0>)


## Summary

All tests completed successfully! The model demonstrates:
- ✓ Correct forward pass computation
- ✓ Proper gradient flow through all parameters
- ✓ Effective timestep and genre conditioning
- ✓ Stable numerical outputs (no NaNs, non-zero variance)
- ✓ Learning capability (can overfit tiny batches)

## Summary

All tests completed successfully! The model demonstrates:
- ✓ Correct forward pass computation
- ✓ Proper gradient flow through all parameters
- ✓ Effective timestep and genre conditioning
- ✓ Stable numerical outputs (no NaNs, non-zero variance)
- ✓ Learning capability (can overfit tiny batches)

# VibeShift: Fixed Implementation Summary

## Critical Issues Fixed

### 1. **MelPatchEmbedding Channel Bug** ✅
**Problem**: `unfold()` operation lost channel information during reshape
**Fix**: Proper permute operation to preserve channels
```python
patches = patches.permute(0, 2, 3, 1, 4, 5).contiguous()  # (B, H', W', C, patch_h, patch_w)
patches = patches.reshape(B, -1, C * self.patch_height * self.patch_width)
```

### 2. **DiT Mel Spectrogram Support** ✅
**Problem**: DiT only worked with pre-patchified linear inputs
**Fix**: 
- Added `MelPatchEmbedding` integration
- Added `use_mel_patches=True` mode
- Takes mel input: `(B, C, n_mels, time_steps)`
- Outputs mel: `(B, C, n_mels, time_steps)`

### 3. **Mel Reconstruction** ✅
**Problem**: No way to convert patches back to mel spectrogram
**Fix**: Proper reshape and permute to reconstruct original mel shape
```python
x = x.reshape(B, num_patches_h, num_patches_w, C, patch_h, patch_w)
x = x.permute(0, 3, 1, 4, 2, 5).contiguous()
x = x.reshape(B, C, H, W)
```

### 4. **TimeEmbedding Integration** ✅
**Problem**: Time information only in FiLM, not added to patches
**Fix**: TimeEmbedding now added to all patches before transformer blocks
```python
t_emb = self.time_emb(t)  # (B, embed_dim)
x = x + t_emb.unsqueeze(1)  # Add to all patches
```

### 5. **Flow Matching Shape Compatibility** ✅
**Problem**: Flow matching used wrong tensor shapes (B, L, D) instead of (B, C, H, W)
**Fix**: Updated `t_expanded` reshaping for mel spectrograms
```python
t_expanded = t.view(batch_size, 1, 1, 1)  # Broadcast to (B, C, H, W)
```

## Architecture Overview

```
Input: Mel Spectrogram (B, 1, 128, 512)
   ↓
MelPatchEmbedding (8×8 patches)
   ↓
Patches (B, 1024, 512) + TimeEmbedding
   ↓
DiTBlock × 12 (RoPE + FiLM + MLP)
   ↓
Output Projection (B, 1024, 64)
   ↓
Reconstruction (reshape + permute)
   ↓
Output: Rock Mel Spectrogram (B, 1, 128, 512)
```

## Complete Pipeline

### Training
```bash
python train.py
```

**Flow Matching Loss**:
- Sample timestep t ~ Uniform(0, 1)
- Interpolate: x(t) = (1-t)·x_source + t·x_rock
- True velocity: v = x_rock - x_source
- Predicted velocity: v_pred = DiT(x(t), t, genre_id)
- Loss: MSE(v_pred, v)

### Inference
```bash
python inference.py --checkpoint checkpoints/vibeshift/best_model.pt \
                    --input my_song.wav \
                    --output my_song_rock.wav \
                    --steps 100 \
                    --method heun
```

**Sampling**:
- Start: x(0) = source mel
- ODE: dx/dt = v(x, t, genre=rock)
- Integrate from t=0 to t=1 using Heun's method
- Result: x(1) = rock mel

## Key Features

✅ **End-to-end mel processing**: Input mel → Output mel  
✅ **Proper channel handling**: No information loss in patches  
✅ **Time conditioning**: Both in TimeEmbedding and FiLM  
✅ **Genre conditioning**: Via FiLM layers in each block  
✅ **Flow matching**: Continuous path from source to rock  
✅ **Flexible sampling**: Euler (fast) or Heun (accurate)  

## Model Configuration

See `configs/dit.yaml`:
- **Patch size**: 8×8 (configurable)
- **Embed dim**: 512
- **Blocks**: 12 transformer layers
- **Attention**: 8 heads with RoPE
- **Genres**: 2 (source, rock)

## Usage Example

```python
from models.dit import DiT
from models.flow import FlowMatching

# Initialize
dit = DiT(use_mel_patches=True, patch_height=8, patch_width=8)
flow = FlowMatching(dit)

# Training
loss = flow(source_mel, rock_mel, genre_ids)

# Inference
rock_mel = flow.sample_heun(source_mel, genre_id=1, num_steps=100)
```

## Next Steps

1. **Prepare paired data**: Source and rock mel spectrograms
2. **Train model**: Run `train.py` with your dataset
3. **Test inference**: Use `inference.py` to transform songs
4. **Fine-tune**: Adjust patch size, steps, learning rate

## Files Modified

- `utills/embedding.py`: Fixed MelPatchEmbedding
- `models/dit.py`: Added mel support + reconstruction
- `models/flow.py`: Fixed shapes for mel spectrograms
- `train.py`: Complete training pipeline (NEW)
- `inference.py`: Audio transformation script (NEW)
